# Tree Trunk Mapper -- Detection Pipeline Demo

**tree-trunk-mapper** detects and maps individual tree trunks from LiDAR point
clouds.  The algorithm works in three stages:

1. **Height slice** -- Extract a thin horizontal band at breast height (1.3 m)
   where trunk cross-sections appear as roughly circular clusters.
2. **DBSCAN clustering** -- Group nearby points in the XY plane so that each
   cluster corresponds to one trunk.
3. **RANSAC cylinder fit** -- Fit a circle (vertical cylinder cross-section) to
   each cluster and derive the diameter at breast height (DBH).

### Use cases

- **Forestry inventory** -- Automatically measure stem positions and diameters
  from terrestrial or aerial LiDAR scans.
- **Autonomous navigation** -- Build a trunk-level map of a forest environment
  for robot localisation and path planning.

---
## 1. Create a Synthetic Forest

In [ ]:
import numpy as np
import open3d as o3d
import pandas as pd
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt

# ---------- ground truth trees ----------
GROUND_TRUTH = [
    # (x, y, radius)
    (0.0,  0.0,  0.12),
    (3.0,  1.0,  0.08),
    (6.0,  0.5,  0.15),
    (1.5,  4.0,  0.10),
    (5.0,  4.5,  0.20),
    (8.0,  3.0,  0.25),
    (2.0,  8.0,  0.09),
    (7.0,  7.5,  0.18),
]

rng = np.random.default_rng(42)


def make_cylinder_points(
    cx, cy, radius, z_min=0.0, z_max=4.0, n_points=800, noise_std=0.005
):
    """Generate noisy points on a vertical cylinder surface."""
    theta = rng.uniform(0, 2 * np.pi, n_points)
    z = rng.uniform(z_min, z_max, n_points)
    x = cx + radius * np.cos(theta) + rng.normal(0, noise_std, n_points)
    y = cy + radius * np.sin(theta) + rng.normal(0, noise_std, n_points)
    return np.column_stack([x, y, z])


all_points = []
for cx, cy, r in GROUND_TRUTH:
    all_points.append(make_cylinder_points(cx, cy, r))

# Ground plane
ground = rng.uniform([-1, -1, -0.05], [10, 10, 0.05], (2000, 3))
all_points.append(ground)

# Random noise points (leaves / clutter)
noise = rng.uniform([-1, -1, 0.0], [10, 10, 5.0], (500, 3))
all_points.append(noise)

forest_pts = np.vstack(all_points)
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(forest_pts.astype(np.float64))

# Save as PCD
pcd_path = Path("synthetic_forest.pcd")
o3d.io.write_point_cloud(str(pcd_path), pcd)
print(f"Saved {len(forest_pts):,} points to {pcd_path}")

# Print ground truth table
gt_df = pd.DataFrame(GROUND_TRUTH, columns=["x", "y", "radius"])
gt_df["dbh"] = gt_df["radius"] * 2
gt_df.index.name = "tree_id"
print("\nGround truth:")
gt_df

---
## 2. Single-Frame Detection

In [ ]:
from tree_trunk_mapper.detector import detect_trunks

detections = detect_trunks(
    pcd,
    slice_height=1.3,
    slice_thickness=0.3,
    eps=0.2,
    min_samples=15,
    ransac_iterations=2000,
    inlier_threshold=0.03,
    min_radius=0.02,
    max_radius=0.50,
    min_inlier_ratio=0.3,
)

print(f"Detected {len(detections)} trunks (ground truth: {len(GROUND_TRUTH)})\n")

det_rows = []
for i, d in enumerate(detections):
    det_rows.append({
        "det_id": i,
        "x": round(d.center[0], 3),
        "y": round(d.center[1], 3),
        "z": round(d.center[2], 3),
        "radius": round(d.radius, 4),
        "dbh": round(d.dbh, 4),
        "inlier_count": d.inlier_count,
    })
det_df = pd.DataFrame(det_rows)
print("Detection results:")
display(det_df)

# --- Match detections to ground truth ---
from scipy.spatial import KDTree

gt_xy = np.array([[cx, cy] for cx, cy, _ in GROUND_TRUTH])
det_xy = np.array([[d.center[0], d.center[1]] for d in detections])

if len(det_xy) > 0:
    tree = KDTree(gt_xy)
    dists, idxs = tree.query(det_xy)
    print("\nComparison with ground truth:")
    comp_rows = []
    for di, (dist, gi) in enumerate(zip(dists, idxs)):
        gt_r = GROUND_TRUTH[gi][2]
        comp_rows.append({
            "det_id": di,
            "matched_gt": gi,
            "pos_error_m": round(dist, 4),
            "gt_dbh": round(2 * gt_r, 4),
            "det_dbh": round(detections[di].dbh, 4),
            "dbh_error_m": round(abs(detections[di].dbh - 2 * gt_r), 4),
        })
    comp_df = pd.DataFrame(comp_rows)
    display(comp_df)

---
## 3. Visualise Detections

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Top-down view (XY) ---
ax = axes[0]
ax.set_title("Top-down view (XY)")
ax.scatter(forest_pts[:, 0], forest_pts[:, 1], s=0.3, c="0.85", alpha=0.4, rasterized=True)

for i, (cx, cy, r) in enumerate(GROUND_TRUTH):
    circle = plt.Circle((cx, cy), r, fill=False, edgecolor="green", linewidth=1.5, linestyle="--")
    ax.add_patch(circle)
    ax.annotate(f"GT{i}", (cx, cy), fontsize=7, ha="center", va="bottom", color="green")

for i, d in enumerate(detections):
    circle = plt.Circle((d.center[0], d.center[1]), d.radius, fill=False, edgecolor="red", linewidth=1.5)
    ax.add_patch(circle)
    ax.annotate(f"D{i}", (d.center[0], d.center[1]), fontsize=7, ha="center", va="top", color="red")

ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_aspect("equal")
ax.legend(
    handles=[
        plt.Line2D([], [], color="green", linestyle="--", label="Ground truth"),
        plt.Line2D([], [], color="red", label="Detection"),
    ]
)

# --- Side view (XZ) showing height slice ---
ax = axes[1]
ax.set_title("Side view (XZ) with height slice")
ax.scatter(forest_pts[:, 0], forest_pts[:, 2], s=0.3, c="0.85", alpha=0.4, rasterized=True)

ground_z = np.percentile(forest_pts[:, 2], 5)
slice_lo = ground_z + 1.3 - 0.15
slice_hi = ground_z + 1.3 + 0.15
ax.axhspan(slice_lo, slice_hi, color="orange", alpha=0.25, label="Height slice (1.3 m)")

for d in detections:
    ax.plot(d.center[0], d.center[2], "rx", markersize=8)

ax.set_xlabel("X (m)")
ax.set_ylabel("Z (m)")
ax.legend()

fig.tight_layout()
plt.show()

---
## 4. Multi-Frame Mapping

In [ ]:
from tree_trunk_mapper.mapper import TrunkMapper

mapper = TrunkMapper(merge_radius=0.5)

# Create 3 frames with slight viewpoint offsets (simulating a moving sensor)
offsets = [
    np.array([0.0, 0.0, 0.0]),
    np.array([0.05, -0.03, 0.0]),
    np.array([-0.04, 0.06, 0.0]),
]

for frame_idx, offset in enumerate(offsets):
    # Shift point cloud slightly to simulate viewpoint change
    shifted_pts = forest_pts + offset + rng.normal(0, 0.002, forest_pts.shape)
    frame_pcd = o3d.geometry.PointCloud()
    frame_pcd.points = o3d.utility.Vector3dVector(shifted_pts.astype(np.float64))

    frame_dets = detect_trunks(
        frame_pcd,
        slice_height=1.3,
        slice_thickness=0.3,
        eps=0.2,
        min_samples=15,
        ransac_iterations=2000,
        min_inlier_ratio=0.3,
    )
    mapper.add_detections(frame_dets)
    print(f"Frame {frame_idx}: {len(frame_dets)} detections")

trunk_map = mapper.get_map()
print(f"\nMerged map: {len(trunk_map)} unique trunks")

map_rows = []
for t in trunk_map:
    map_rows.append({
        "trunk_id": t.trunk_id,
        "x": round(t.position[0], 3),
        "y": round(t.position[1], 3),
        "dbh": round(t.dbh, 4),
        "observations": t.observation_count,
    })
pd.DataFrame(map_rows)

---
## 5. Export Results

In [ ]:
from tree_trunk_mapper.export import export_geojson, export_csv

geojson_path = Path("trunk_map.geojson")
csv_path = Path("trunk_map.csv")

export_geojson(trunk_map, geojson_path)
export_csv(trunk_map, csv_path)

print(f"Exported GeoJSON -> {geojson_path}")
print(f"Exported CSV     -> {csv_path}\n")

# Display the CSV as a DataFrame
result_df = pd.read_csv(csv_path)
result_df

---
## 6. Accuracy Analysis

In [ ]:
# Match mapped trunks to ground truth
map_xy = np.array([[t.position[0], t.position[1]] for t in trunk_map])
gt_tree = KDTree(gt_xy)
map_dists, map_idxs = gt_tree.query(map_xy)

# Only keep matches within 0.5 m
match_mask = map_dists < 0.5
matched_dists = map_dists[match_mask]
matched_gt_idxs = map_idxs[match_mask]
matched_map_idxs = np.where(match_mask)[0]

gt_dbh_matched = np.array([2 * GROUND_TRUTH[gi][2] for gi in matched_gt_idxs])
det_dbh_matched = np.array([trunk_map[mi].dbh for mi in matched_map_idxs])
dbh_errors = np.abs(det_dbh_matched - gt_dbh_matched)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Position error histogram
ax = axes[0]
ax.hist(matched_dists, bins=10, edgecolor="black", alpha=0.7, color="steelblue")
ax.set_title("Position Error")
ax.set_xlabel("Error (m)")
ax.set_ylabel("Count")
ax.axvline(np.mean(matched_dists), color="red", linestyle="--",
           label=f"Mean = {np.mean(matched_dists):.3f} m")
ax.legend()

# DBH error
ax = axes[1]
ax.bar(range(len(dbh_errors)), dbh_errors * 100, color="coral", edgecolor="black", alpha=0.7)
ax.set_title("DBH Error per Matched Trunk")
ax.set_xlabel("Match index")
ax.set_ylabel("DBH error (cm)")
ax.axhline(np.mean(dbh_errors) * 100, color="red", linestyle="--",
           label=f"Mean = {np.mean(dbh_errors)*100:.2f} cm")
ax.legend()

# Detection rate summary
ax = axes[2]
n_gt = len(GROUND_TRUTH)
n_detected = len(matched_dists)
n_false_pos = int((~match_mask).sum())
categories = ["Ground truth", "Detected", "Missed", "False +"]
values = [n_gt, n_detected, n_gt - n_detected, n_false_pos]
colours = ["steelblue", "seagreen", "salmon", "orange"]
ax.bar(categories, values, color=colours, edgecolor="black", alpha=0.8)
ax.set_title("Detection Summary")
ax.set_ylabel("Count")
for i, v in enumerate(values):
    ax.text(i, v + 0.15, str(v), ha="center", fontweight="bold")

fig.tight_layout()
plt.show()

# Print summary
print(f"Detection rate : {n_detected}/{n_gt} = {n_detected/n_gt*100:.0f}%")
print(f"False positives: {n_false_pos}")
print(f"Mean position error : {np.mean(matched_dists):.4f} m")
print(f"Mean DBH error      : {np.mean(dbh_errors)*100:.2f} cm")

---

### Cleanup

Remove temporary files created during the demo.

In [ ]:
for p in [pcd_path, geojson_path, csv_path]:
    p.unlink(missing_ok=True)
print("Temporary files removed.")